# Introduction to Keras
In this notebook, we will begin our introduction to keras.   We will follow Chapter 10 of **Hands On Machine Learning** for most of this, with some slight changes.

We will build a simple model to start with, implementing a model to classify MNIST digits.

NOTE: You can ignore Tensorflow/CUDA/GPU-related warnings below.  These go away if you run on a machine with GPUs, which we will do later in the course.

In [34]:
import tensorflow as tf
from tensorflow import keras
print(tf.__version__)
print(keras.__version__)
#
# Do these to reduce arning messages below
tf.get_logger().setLevel('ERROR')
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook+pdf'
import time

t00 = time.time()


2.19.1
3.10.0


## Get the data
We will use our MNIST data sample yet again!   This time, we will use the version that comes along prepackaged with the keras package.

Keras has a small number of datasets included as part of the package (see [here](https://keras.io/datasets/) for more details)   These include:
1.  MNIST:  60,000 28x28 grayscale images of the 10 digits, along with a test set of 10,000 images.
2.  Reuters newswire topics classification:  11,228 newswires from Reuters, labeled over 46 topics, for text processing and classification. 
3.  CIFAR10 small image classification: Dataset of 50,000 32x32 color training images, labeled over 10 categories, and 10,000 test images.   There is a similar dataset (CIFAR100) with 100 labeled catagories.

Below we load the MNIST dataset (both training and test).   

Due to time considerations, we will use the smaller train set (7000 out of the available 60,000) but we use all 10,000 of the test set.

We also use the **keras** method of loading the data, versus reading the data from csv files as we have in the past.

In [35]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
print("Train info all:",train_images.shape, train_labels.shape)
print("Test info all:",test_images.shape, test_labels.shape)

short = True
if short:
    train_images = train_images[:7000,:]
    train_labels = train_labels[:7000]
    # test_images = test_images[:3000,:]
    # test_labels = test_labels[:3000]
#
print()
print("Train info used:",train_images.shape, train_labels.shape)
print("Test info used:",test_images.shape, test_labels.shape)

print()
print("Python type of the images:",type(train_images))
print("Python type of the labels:",type(train_labels))

print()
print("Python shape of the images:",train_images.shape)
print("Python shape of the labels:",train_labels.shape)

print()
print("Count of unique train labels:",np.unique(train_labels, return_counts=True))
print("Count of unique test labels: ",np.unique(test_labels, return_counts=True))



Train info all: (60000, 28, 28) (60000,)
Test info all: (10000, 28, 28) (10000,)

Train info used: (7000, 28, 28) (7000,)
Test info used: (10000, 28, 28) (10000,)

Python type of the images: <class 'numpy.ndarray'>
Python type of the labels: <class 'numpy.ndarray'>

Python shape of the images: (7000, 28, 28)
Python shape of the labels: (7000,)

Count of unique train labels: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([691, 784, 675, 716, 716, 610, 709, 754, 650, 695]))
Count of unique test labels:  (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([ 980, 1135, 1032, 1010,  982,  892,  958, 1028,  974, 1009]))


## Prepare the feature data
We need to make sure the feature data is normalized.  

Since we know our max and min is 255/0, we can just divide each pixel by 255.

In [36]:

train_images = train_images.astype('float32')/255
test_images = test_images.astype('float32')/255

## Prepare the label data
The labels run from 0-9, but we need to make them 1-hot.   Why?  From our discussion of neural networks, we know we need a separate output for each class in our data.

We use a keras utility to do this.

In [37]:

train_labels_cat = keras.utils.to_categorical(train_labels)
test_labels_cat = keras.utils.to_categorical(test_labels)


# Exercise 1:
Print out the first 20 rows of the labels and labels_cat in the train data to make sure you understand what "to_categorical" does.  (Should need no more than two print statements to do this!)

Look at the digit value printed out for the train_labels, and then match them visually with the corresponding train_labels_cat.

In [38]:
# your code here
print("Original labels:", train_labels[:20])
print("One-hot encoded labels:", train_labels_cat[:20])

Original labels: [5 0 4 1 9 2 1 3 1 4 3 5 3 6 1 7 2 8 6 9]
One-hot encoded labels: [[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


## Build the Model
Our model will be just like the one we described in the notes:
1. An input layer, 784(=28x28) features wide.  Why 28x28?  This is our image size!
2. A hidden layer, 100 "nodes" wide, using the "tanh" activation function.  Why 100?  Just a guess!!
3. An output layer, 10 modes wide, using the softmax activation function.  Why 10?   We have 10 digits and need 10 outputs!

Building this model with keras is quite simple.

In [39]:

model = keras.models.Sequential()
model.add(keras.layers.Input(shape=[28, 28]))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(100,activation='tanh'))
model.add(keras.layers.Dense(10,activation='softmax'))

## Compile the model
Compiling the model is necessary before you can train the model.  Compiling configures the learning process and sets some other hyperparameters:
1.  A loss function. This is the objective that the model will try to minimize. There are a range of choices which can be examined [here](https://keras.io/losses/).   For classification problems the typical choices are:
    * categorical_crossentropy: used for multi-class classification (like MNIST)
    * binary_crossentropy: used for binary classification (like any one vs all problem, or a problem with just one signal and one background like our pulsar problem earlier)
2.  An optimizer. This controls how the minimum of the loss function is found.   SGD (stochastic gradient descent) is typical, as is Adam (see [here](https://arxiv.org/abs/1412.6980v8) for more details).   The text has a good discussion of these and many others in Chapter 11 (see p. 351).   The optimizers availble in Keras are discussed [here](https://keras.io/api/optimizers/).  We will use Adam.
3.  A list of metrics. For any classification problem you will usually want to set this to metrics=['accuracy']. 

Another thing we do below is to save the weights of the compiled model right after we first compile it.   These weights are initiailized to some random (and typically small) values.   This will be useful if we end up calling the model in an optimization loop later.  For now, just make sure you do this.

**NOTICE!!**  There are ALOT of **trainable** parameters in this very simple and "small" neural network model!

In [40]:

model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
print(model.summary())

#
# If we reload this right before fitting the model, the model will start from scratch
model.save_weights('model_init.weights.h5')


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

None


# Exercise 2
Calculate the number of trainable paremters based on the class notes, and make sure you get the number list above from printing the model summary.   Describe your calculation in the next **Markdown** window.


In [41]:
print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

None



**Explanation**: 

Your explanation here:

To verify the number of trainable parameters in our neural network model, I begin by analyzing the architecture. The model includes a `Flatten` layer, a `Dense` hidden layer with 100 units using a `tanh` activation function, and an output `Dense` layer with 10 units using a `softmax` activation function. The `Flatten` layer reshapes the 28×28 input images into vectors of length 784 and does not contribute any trainable parameters.

The first `Dense` layer connects all 784 input features to 100 neurons. Each of these connections involves a weight, and each neuron also has a bias term. Therefore, the total number of trainable parameters in this layer is calculated as 784 × 100 (weights) + 100 (biases) = 78,500.

The second `Dense` layer connects the 100 outputs from the previous layer to 10 output neurons. Similarly, each of these 10 neurons has 100 weights and 1 bias term, resulting in 100 × 10 + 10 = 1,010 trainable parameters.

Adding both layers together, the total number of trainable parameters in the model is 78,500 + 1,010 = 79,510. This matches exactly with the number reported by `model.summary()`, confirming that the calculation is correct and consistent with the model architecture.



## Fitting the model
The "fit" method takes the following arguments:
1.  The input features: in our case this is "train_images".
2.  The output labels: input case this is the 1-hot "train_labels_cat"
3.  The number of epochs to run.  Remember that an "epoch" is defined as an iteration in which the entire set of training samples has been passed through the model.   We use 50 below, but it is important to choose a number large enough that your performance (on the test set!) converges.  Remember of discussion of under vs over training a model.   We will find that we might not want to use ALL of the epochs we give to the "fit" method - this is called "early stopping".   More on this below.
4.  The batch size: this is the number of training samples that are passed through the model before the weights are updated.  Note the difference between this and the number of "epochs".  We will use a batch size of 128 (typically). A good discussion of the issues surrounding batch size and epochs is found [here] (https://stats.stackexchange.com/questions/164876/tradeoff-batch-size-vs-number-of-iterations-to-train-a-neural-network).   A batch size larger than 1 speeds up training.
5.  An **optional** validation set.   This is a set of features and labels that are used to assess the performance of the model during the fit, at the end of each epoch.   Statistics on this (and the training set) are collected and returned when the fit is finished.

The fit returns a **history** object, containing a .history dictionary with the following entries:
*  history.history\['loss'\]: A list of the values of the loss function (evaluated on the training sample) at the end of each epoch, ordered by epoch.
*  history.history\['accuracy'\]: A list of the values of the accuracy (evaluated on the training sample) at the end of each epoch, ordered by epoch.
*  history.history\['val_loss'\]: A list of the values of the loss function (evaluated on the validation sample) at the end of each epoch, ordered by epoch.  Only returned if a validation sample is supplied.
*  history.history\['val_accuracy'\]: A list of the values of the accuracy (evaluated on the validation sample) at the end of each epoch, ordered by epoch.  Only returned if a validation sample is supplied.



## Training vs Validation vs Testing
You may have noticed that we have (re-)introduced the "validation" sample.   This is sometimes confused with the "testing" sample, but they are different.

To be clear:
1.  **Training set**: A set of examples used for learning, that is to **fit** the values parameters (weights) of the classifier.

2.  **Validation set**: A set of examples used to **tune**  the parameters (for example the number of nodes in the hidden layer) of a classifier.

3.  **Test set**: A set of examples used only to assess the performance of an already fit classifier.

If we do k-fold validation, we typically have **no test set** .   We split the training set up into k-folds, train on each fold and average the results.   Do this many times to choose our parameter setting (like the number of hidden nodes).   Once finished, we retrain our model using the **full** training sample.   Our expected performance is the average performance using the k-folds (at the parameter setting we chose).

For this MNIST data sample we will do something slightly different, since we have a large available training set:
1.  We will use the MNIST **training** sample to supply data for our k-fold validation process - meaning this sample will be broken up into training and validation.
2.  We will use the MNIST **testing** sample (which is actually a sample independent of the training sample) to test our fully trained sample, after k-fold validation.

In the example fit below, we use the MNIST **training** sample, and split it into a single **temporary** training sample and a separate **validation** set in the "fit" function.  We will use the **test** sample from above separately.

In [42]:
from sklearn.model_selection import train_test_split
train_images_temp,val_images,train_labels_cat_temp,val_labels_cat = train_test_split(train_images,train_labels_cat, 
                                                                                         test_size=0.1, random_state=42)


In [43]:
#
# We do this load of the initial weights to make sure we start the training from scratch
model.load_weights('model_init.weights.h5')
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
# Now actually train
results = model.fit(train_images_temp,train_labels_cat_temp,epochs=50,batch_size=128,validation_data=(val_images,val_labels_cat))

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6003 - loss: 1.4069 - val_accuracy: 0.8700 - val_loss: 0.5202
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8836 - loss: 0.4405 - val_accuracy: 0.9043 - val_loss: 0.3899
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9080 - loss: 0.3445 - val_accuracy: 0.9100 - val_loss: 0.3423
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9231 - loss: 0.2752 - val_accuracy: 0.9229 - val_loss: 0.3144
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9332 - loss: 0.2517 - val_accuracy: 0.9229 - val_loss: 0.2982
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9366 - loss: 0.2289 - val_accuracy: 0.9243 - val_loss: 0.2866
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9495 - loss: 0.1921 - val_accuracy: 0.9271 - val_loss: 0.2787
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9557 - loss: 0.1761 - val_accuracy: 0.9243 - val_loss:

# What is in the history object?

We print the history object and things that are inside of it below.

In [44]:
print()
print("Just the results object")
print(results)

print()
print("Just the results.history object - this is a dictionary")
print(results.history)

print()
print("All of the keys in results.history dictionary")
print(results.history.keys())

print()
print("Just the val_accuracy part of the results.history object - this is a list")
print(results.history['val_accuracy'])


Just the results object

Just the results.history object - this is a dictionary
{'accuracy': [0.7522222399711609, 0.8934920430183411, 0.9114285707473755, 0.9242857098579407, 0.9330158829689026, 0.9415873289108276, 0.9452381134033203, 0.9530158638954163, 0.9561904668807983, 0.9599999785423279, 0.9622222185134888, 0.9677777886390686, 0.9715873003005981, 0.9746031761169434, 0.9774603247642517, 0.9806349277496338, 0.9817460179328918, 0.9834920763969421, 0.9841269850730896, 0.988095223903656, 0.989047646522522, 0.9915872812271118, 0.9917460083961487, 0.992222249507904, 0.993492066860199, 0.9944444298744202, 0.9953968524932861, 0.9965079426765442, 0.996666669845581, 0.997936487197876, 0.9984126687049866, 0.9987301826477051, 0.9988889098167419, 0.9987301826477051, 0.9993650913238525, 0.9996825456619263, 0.9996825456619263, 0.9998412728309631, 0.9998412728309631, 0.9998412728309631, 0.9998412728309631, 0.9998412728309631, 0.9998412728309631, 1.0, 0.9998412728309631, 0.9998412728309631, 1.0, 1

# Putting the history object in a dataframe

This will make it easy to print AND plot the loss, accuracy, etc vs epoch 

In [45]:
import pandas as pd
df = pd.DataFrame(results.history)
print(df.head(10))

   accuracy      loss  val_accuracy  val_loss
0  0.752222  0.952607      0.870000  0.520229
1  0.893492  0.411234      0.904286  0.389942
2  0.911429  0.324846      0.910000  0.342261
3  0.924286  0.276734      0.922857  0.314354
4  0.933016  0.247600      0.922857  0.298180
5  0.941587  0.222465      0.924286  0.286577
6  0.945238  0.199861      0.927143  0.278723
7  0.953016  0.181979      0.924286  0.272652
8  0.956190  0.165176      0.924286  0.267101
9  0.960000  0.150763      0.920000  0.265764


## Saving a model
Once we have trained our model, we are ready to use it.  However, it often takes a long time to train a model, once trained we may want to use it at a different time (and using a different python program).   Retraining the model is not practical.  

Instead, we will often save the model immediately upon training it, so we can simply **load** the already trained model into memory the next time we want to use it.

In [46]:
model.save('fully_trained_model.keras')  # creates a keras formatted file 'fully_trained_model.keras' in the current directory

## Examine performance
First let's look at the returned history object:

In [47]:
import pandas as pd
import numpy as np
import plotly.express as px

training_vals_acc = results.history['accuracy']
training_vals_loss = results.history['loss']
valid_vals_acc = results.history['val_accuracy']
valid_vals_loss = results.history['val_loss']
iterations = len(training_vals_acc)
print("Number of iterations:",iterations)
#
# MAKE A DATAFRAME OF RESULTS TO DISPLAY AND PLOT
df = pd.DataFrame(results.history)
#
# A trick to get the epoch into our dataframe
df['epoch'] = df.index + 1
display(df.style)


Number of iterations: 50


,accuracy,loss,val_accuracy,val_loss,epoch
0,0.752222,0.952607,0.870000,0.520229,1
1,0.893492,0.411234,0.904286,0.389942,2
2,0.911429,0.324846,0.910000,0.342261,3
3,0.924286,0.276734,0.922857,0.314354,4
4,0.933016,0.247600,0.922857,0.298180,5
5,0.941587,0.222465,0.924286,0.286577,6
6,0.945238,0.199861,0.927143,0.278723,7
7,0.953016,0.181979,0.924286,0.272652,8
8,0.956190,0.165176,0.924286,0.267101,9
9,0.960000,0.150763,0.920000,0.265764,10


## Plotting Performance
Here we look at the train and validation performance versus epoch.

In [48]:
#
# ACCURACY
fig = px.line(df, x='epoch', y=['accuracy','val_accuracy'], title='Accuracy vs Epoch')
fig.show("plotly_mimetype")
#
# Loss
fig = px.line(df, x='epoch', y=['loss','val_loss'], title='Loss vs Epoch')
fig.show("plotly_mimetype")

## Loading a pre-trained model
Here we will load the pretrained model (deleting the version in memory to prove that this works!), and then apply this model to unseen data - our testing sample the we loaded above.

To get the model performance, we have two options:
1.  model.evaluate: This we use if we have labeled samples.   It returns the overall loss, as well as the calculated accuracy on that labeled dataset.
2.  model.predict:  This can be used on labeld or unlabeled data.  It returns the output of the model (in our case the 10 probabilities for the 10 classes) for **each** sample.  If you do have labeled data, you can compare the predicted output to the known label.

NOTE: we are running this on the **test** samples we read in above: test_images, test_labels, **not** the validation samples we used in the fit call val_images,val_labels_cat.

In [49]:
import numpy as np
    
# returns a compiled model
# identical to the previous one (note the new name!!)
trained_model = keras.models.load_model('fully_trained_model.keras')
#
# Get the overall performance for the test sample
test_loss, test_acc = trained_model.evaluate(test_images,test_labels_cat)
print("Test sample loss: ",test_loss, "; Test sample accuracy: ",test_acc)
#
# Get the individual predictions for each sample in the test set
predictions = trained_model.predict(test_images)
#
# Get the max probabilites for each rows
test_probs = np.max(predictions, axis = 1)
#
# Get the predicted classes for each row
test_preds = np.argmax(predictions, axis = 1)
#
# Now loop over the first twenty samples and compare truth to prediction
print("Label\t Pred\t Prob")
for num,(label,cl,pr) in enumerate(zip(test_labels,test_preds,test_probs)):
    print(label,'\t',cl,'\t',round(pr,3))
    if num>20:
        break


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9287 - loss: 0.2666  
Test sample loss:  0.2424459457397461 ; Test sample accuracy:  0.9373000264167786
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 765us/step
Label	 Pred	 Prob
7 	 7 	 1.0
2 	 2 	 1.0
1 	 1 	 0.999
0 	 0 	 1.0
4 	 4 	 0.995
1 	 1 	 1.0
4 	 4 	 1.0
9 	 9 	 0.999
5 	 6 	 0.909
9 	 9 	 0.998
0 	 0 	 0.999
6 	 6 	 0.991
9 	 9 	 1.0
0 	 0 	 1.0
1 	 1 	 1.0
5 	 5 	 0.99
9 	 9 	 1.0
7 	 7 	 1.0
3 	 3 	 0.915
4 	 4 	 1.0
9 	 9 	 0.992
6 	 6 	 0.999


## Early Stopping
Notice that in the loss plot above, the model performance was best somewhere in the range of epochs 10-20, yet we continued to train the model until epoch 50.   

Keras makes it possible to do two things:
1.  Stop the training once a condition has been met, using a module called "EarlyStopping".   This has two parameters:
   * what is monitored for stopping: we will use 'val_loss' the loss in the validation set.
   * "patience": this is how many epochs to wait after the condition has been met.  The idea being that there are fluctuations in the parameter you are monitoring, and you don't want to stop if you just had a small downward fluctuation.   So you wait to see if the performance does not get better.
2.  Save the best model prior to stopping, using a module called "ModelCheckpoint".   You tell this module what to monitor, and every time the condition is met, you write out (and overwrite the previous) a new file containing the full model info.

In [50]:
#
# Define the model here - call the model "model"
hidden_nodes = 100
activation = 'tanh'
optimizer = 'adam'
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=[28, 28]))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(hidden_nodes,activation=activation))
model.add(keras.layers.Dense(10,activation='softmax'))
#
# Now compile the model
model.compile(optimizer=optimizer,loss='categorical_crossentropy',metrics=['accuracy'])
#
# If we reload this right before fitting the model, the model will start from scratch
model.save_weights('model_init.weights.h5')
#
# Define the callbacks
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
             keras.callbacks.ModelCheckpoint(filepath='best_model.keras', monitor='val_loss', save_best_only=True)]
#
# Now load the initial weights we saved above - this **ensures** the model starts training from scratch
model.load_weights('model_init.weights.h5')
#
# Now fit the model
results = model.fit(train_images_temp,train_labels_cat_temp,
                              epochs=50,
                              batch_size=128,
                              verbose=1, # set to 0 for no printout while running
                              callbacks=callbacks, # Early stopping
                              validation_data=(val_images,val_labels_cat))
#
# NOTE: We DON"T save the model, because that was handled by the callback!!!
#
# get performance info from the results dictionary
training_vals_acc = results.history['accuracy']
training_vals_loss = results.history['loss']
valid_vals_acc = results.history['val_accuracy']
valid_vals_loss = results.history['val_loss']
iterations = len(training_vals_acc)
print("Number of iterations:",iterations)
print("Epoch\t Train Loss\t Train Acc\t Val Loss\t Val Acc")
i = 0
for tl,ta,vl,va in zip(training_vals_loss,training_vals_acc,valid_vals_loss,valid_vals_acc):
    print(i,'\t',round(tl,5),'\t',round(ta,5),'\t',round(vl,5),'\t',round(va,5))
    i += 1


Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6132 - loss: 1.3703 - val_accuracy: 0.8671 - val_loss: 0.5050
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8876 - loss: 0.4273 - val_accuracy: 0.9014 - val_loss: 0.3799
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9144 - loss: 0.3293 - val_accuracy: 0.9100 - val_loss: 0.3333
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9256 - loss: 0.2771 - val_accuracy: 0.9200 - val_loss: 0.3078
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9394 - loss: 0.2445 - val_accuracy: 0.9186 - val_loss: 0.2947
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9414 - loss: 0.2249 - val_accuracy: 0.9229 - val_loss: 0.2822
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9485 - loss: 0.1967 - val_accuracy: 0.9214 - val_loss: 0.2755
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9499 - loss: 0.1831 - val_accuracy: 0.9229 - val_loss:

**NOTICE**: When I ran this, training stopped at 29 epochs for the full sample (and 37 for the short sample), not 50, and the minimum validation sample loss was a epoch 19 (training continued for patience=10 epochs after this minimum to make sure we did not hit yet another minimum).   Note: results might be different for you since the network has a random initialization.

# Exercise 3: Make a confusion matrix
Run the **best** model from above on the test set, and construct a confusion matrix from the results.  Hint: you do **not** have to rerun the training!!

**NOTE** We copy the calc_performance_multi method from module4 to help us!

The code below gives you a good start!

In [51]:

from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
from sklearn import metrics
from sklearn.metrics import auc

from tabulate import tabulate

def calc_performance_multi(y_vals_true, y_vals_pred,labels):
#
# Get the numbers for the confusion matrix
# To get output: cf_matrix[true_label,pred_label]
    cf_matrix = confusion_matrix(y_vals_true, y_vals_pred, labels=labels)
#
# This is a graphic
    cf_disp = ConfusionMatrixDisplay(confusion_matrix=cf_matrix,display_labels=labels)
#
# Make the header row
    header = [""]
    for column_name in labels:
        header.append('Pred:' + str(column_name))
    table = [header]
#
# Now make the rows with the matrix
    for row_name in labels:
        row = ['True:'+str(row_name)]
        for column_name in labels:
            row_index = labels.index(row_name)
            column_index = labels.index(column_name)
            row.append(cf_matrix[row_index,column_index])
        table.append(row)
    # table = [
    #     [ "", "Predicted Class 1", "Precicted Class 0"],
    #     [ "True Class 1", TP, FN ],
    #     [ "True Class 0", FP, TN ]
    # ]
    print_table_type='fancy_grid'
    print_table = tabulate(table, headers='firstrow', tablefmt=print_table_type)
#    print(print_table)
#
# Get the recall, precision, ands F1 for each individual label
# - return both the "string report" (which you can print)
# - and the "dictionary report" (which you can use for averages and so on)
    report = classification_report(y_vals_true,y_vals_pred,digits=4)
    report_dict = classification_report(y_vals_true,y_vals_pred,output_dict=True, digits=4)
#
    results = {"confusionMatrix":cf_matrix,
                    'confusion_matrix_display':cf_disp,
                    'confusion_matrix_print_table':print_table,      
                    "report":report,"report_dict":report_dict}
    return results

#
# Run on the test set
# returns a compiled model
# identical to the previous one (note the new name!!)
trained_model = keras.models.load_model('best_model.keras') # YOU CAN LOAD THIS FROM DISK... WHAT FILENAME SHOULD YOU USE???

#
# Get the overall performance for the test sample
test_loss, test_acc = trained_model.evaluate(test_images,test_labels_cat)
#
# Get the individual predictions for each sample in the test set
predictions = trained_model.predict(test_images)
#
# Remove code below for student version!
#
# Get the max probabilites for each rows
test_probs = np.max(predictions, axis = 1)
#
# Get the predicted classes for each row
test_preds = np.argmax(predictions, axis = 1)

#
# Now call calc performance
labels = [0,1,2,3,4,5,6,7,8,9]
results_test = calc_performance_multi(test_labels, test_preds,labels)

### FILL THIS IN!!!  get from results_test
print("average recall test:   ", results_test['report_dict']['macro avg']['recall'])

print()
print("The Confusion matrix for test data:")
### FILL THIS IN!!!  get from results_test
print(results_test['confusion_matrix_print_table'])


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9225 - loss: 0.2581  
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 763us/step
average recall test:    0.9324624561489102

The Confusion matrix for test data:
╒════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╤══════════╕
│        │   Pred:0 │   Pred:1 │   Pred:2 │   Pred:3 │   Pred:4 │   Pred:5 │   Pred:6 │   Pred:7 │   Pred:8 │   Pred:9 │
╞════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╡
│ True:0 │      955 │        0 │        0 │        4 │        1 │        4 │       11 │        2 │        3 │        0 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┼──────────┤
│ True:1 │        0 │     1114 │        4 │        2 │        1 │        0 │        3 │        2 │        9 │        0 │
├────────┼──────────┼──────────┼──────────┼──────────┼──────────┼─────────

# Defining your model with a function
In the code above we defined our model in what you might call the "usual way": with a series of python statements!

Now we could also put all of those python statements into the **method** (which sort of hides the details) and then call the method.   If we design the method well, we can make the formation of a model flexible, allowing simple choices of:
- neurons per layer
- activation function
- and so on

This does not necessarily help us right now, but this idea will be very useful when it comes to exploring the **hyperparemeter** space for our models.


In [52]:
def build_model(n_neurons=100,activation='tanh',learning_rate=3e-3,input_shape=[28,28],optimizer="adam"):
    model = keras.models.Sequential()
    model.add(keras.layers.Input(shape=[28, 28]))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(n_neurons,activation=activation))
    model.add(keras.layers.Dense(10,activation='softmax'))
    model.compile(optimizer=optimizer,loss='categorical_crossentropy',metrics=['accuracy'])
    return model


# Exercise 4
Use the above function to define a new model with 200 hidden nodes in the hidden layer.  Train the model and plot the test and train accuracy and loss as we did above.

Train for at least 50 epochs, and make sure to include the callbacks for early stopping with a patience=10.

In [53]:
#
# Define the model here - call the model "new_model"
new_model = build_model(n_neurons=200, activation='tanh', optimizer='adam')
new_model.save_weights('new_model_init.weights.h5')

#
# Define the EarlyStopping and ModelCheckpoint callbacks
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10), keras.callbacks.ModelCheckpoint(filepath='new_best_model.keras',  monitor='val_loss', save_best_only=True)]

#
# Actually fit the model
new_model.load_weights('new_model_init.weights.h5')
new_results = new_model.fit(train_images_temp, 
                           train_labels_cat_temp,
                           epochs=50,
                           batch_size=128,
                           verbose=1,
                           callbacks=callbacks,
                           validation_data=(val_images, val_labels_cat))


Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6474 - loss: 1.2190 - val_accuracy: 0.8886 - val_loss: 0.4240
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9070 - loss: 0.3552 - val_accuracy: 0.9129 - val_loss: 0.3541
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9162 - loss: 0.3055 - val_accuracy: 0.9143 - val_loss: 0.3187
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9358 - loss: 0.2475 - val_accuracy: 0.9243 - val_loss: 0.2984
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9401 - loss: 0.2231 - val_accuracy: 0.9271 - val_loss: 0.2847
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9498 - loss: 0.1844 - val_accuracy: 0.9186 - val_loss: 0.2777
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9563 - loss: 0.1639 - val_accuracy: 0.9343 - val_loss: 0.2734
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9557 - loss: 0.1700 - val_accuracy: 0.9157 - val_loss:

In [54]:

#
# MAKE A DATAFRAME OF RESULTS TO DISPLAY AND PLOT
df = pd.DataFrame(new_results.history)
df['epoch'] = df.index + 1
display(df.style)

#
# ACCURACY
fig = px.line(df, x='epoch', y=['accuracy','val_accuracy'], title='Accuracy vs Iteration')
fig.show("plotly_mimetype")
#
# Loss
fig = px.line(df, x='epoch', y=['loss','val_loss'], title='Loss vs Iteration')
fig.show("plotly_mimetype")

,accuracy,loss,val_accuracy,val_loss,epoch
0,0.783333,0.789836,0.888571,0.423956,1
1,0.906032,0.350843,0.912857,0.354059,2
2,0.917778,0.291233,0.914286,0.318745,3
3,0.935079,0.246142,0.924286,0.298364,4
4,0.942064,0.216901,0.927143,0.284729,5
5,0.946825,0.194341,0.918571,0.277688,6
6,0.950317,0.177145,0.934286,0.273430,7
7,0.958095,0.159017,0.915714,0.271723,8
8,0.963333,0.142055,0.925714,0.265837,9
9,0.965714,0.130299,0.931429,0.255936,10


# Hyperparameter Tuning with Keras Tuner
When we decided to use 1 hidden layer and 100 (or 200) neurons in that layer, we did not really explain **why** we chose those values.  Now we know we could use k-fold validation to choose these hyperparameters, but it turns out there is a built in tool for optimization called Keras Tuner.

Keras Tuner is a library from the TensorFlow team that helps you find the optimal set of hyperparameters for your Keras models. Think of it as an automated assistant that saves you from the tedious process of manual trial-and-error.

In simple terms, it does three main things:

1) Defines a Search Space: Instead of hard-coding values like the number of layers or the learning rate, you define a range or a list of choices for the tuner to explore.

2) Automates the Search: It intelligently tests different combinations of these hyperparameters by repeatedly building, training, and evaluating your model. It supports various strategies like Random Search, Bayesian Optimization, and Hyperband, which are often more efficient than a simple grid search.

3) Finds the Best Model: After the search is complete, it tells you which combination of hyperparameters performed the best on your validation data, allowing you to build and train a final, optimized model.

Essentially, Keras Tuner automates the process of tuning your neural network to achieve the best possible performance.

Note by default, **Keras Tuner does not use k-fold cross-validation**.  Instead, it relies on a single, fixed validation set to evaluate the performance of different hyperparameter combinations.  While it's not a built-in feature, you can manually implement k-fold cross-validation by wrapping the tuning process in a loop. This gives you a more robust evaluation of your hyperparameters at the cost of significantly more computation time.


# Keras Tuner Step 1

We need to make a build function that looks like the code below:

In [55]:
def build_model_tuner(hp):
    """
    Builds a Keras model with hyperparameters for Keras Tuner.
    """
#
# This is the tuning sections -define all of the parameters you want to tune
#
    # Tune the number of neurons in the dense layer
    hp_n_neurons_h1 = hp.Int('n_neurons_h1', min_value=32, max_value=256, step=32)
    
    # Tune the activation function
    hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
        
    # Tune the learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    # Tune the optimizer
    hp_optimizer = hp.Choice('optimizer', values=['adam', 'rmsprop', 'sgd'])
#
# Select optimizer based on the choice
    if hp_optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    elif hp_optimizer == 'rmsprop':
        optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=hp_learning_rate)
  
#
# Define the model and layers
    input_shape=[28,28]
    model = keras.models.Sequential()
#
# These 
    model.add(keras.layers.Input(shape=input_shape))
    model.add(keras.layers.Flatten())
#
# This is the single hidden layer
    model.add(keras.layers.Dense(units=hp_n_neurons_h1, activation=hp_activation))
#
# This is the output layer
    model.add(keras.layers.Dense(10, activation='softmax'))     
    model.compile( optimizer=optimizer,loss='categorical_crossentropy',metrics=['accuracy'])
#
# Return the model
    return model

# Keras Tuner Step 2

We then make a "tuner object" which uses our build function to search over all of the hyperparemeters we defined in the above build function.

NOTE: We still use the EarlyStopping callback, but we removed the ModelCheckpoint as there is no reason to write out the best model until after our tuning is done.

See the code below:

In [56]:
import keras_tuner as kt
# We pass the model-building function directly to the tuner
tuner = kt.RandomSearch(
    build_model_tuner,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=2,
    directory='keras_tuner_dir',
    project_name='mnist_tuning',  # Use a new project name for the new model
    overwrite=True
)

#
# Define the EarlyStopping callbacks; don't use ModelCheckpoint for tuning
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)]

print("\nStarting Keras Tuner search...")
# Use the test set as the validation data for the search
tuner.search(train_images_temp,train_labels_cat_temp, 
                                epochs=10,
                                batch_size=128,
                                verbose=1, # set to 0 for no printout while running
                                callbacks=callbacks, # Early stopping
                                validation_data=(val_images,val_labels_cat))



Trial 10 Complete [00h 00m 05s]
val_loss: 0.2552020400762558

Best val_loss So Far: 0.21410920470952988
Total elapsed time: 00h 00m 52s


# Keras Tuner Step 3: Print out Tuner Results

In [57]:
# Summarize and Display the Results ---

print("\n--- Keras Tuner Search Complete ---")
tuner.results_summary()

# Retrieve the Best Hyperparameters and Train the Final Model ---

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n--- Best Hyperparameters Found ---")
print(f"""
Number of Neurons: {best_hps.get('n_neurons_h1')}
Activation Function: {best_hps.get('activation')}
Optimizer: {best_hps.get('optimizer')}
Learning Rate: {best_hps.get('learning_rate')}
""")



--- Keras Tuner Search Complete ---
Results summary
Results in keras_tuner_dir/mnist_tuning
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 01 summary
Hyperparameters:
n_neurons_h1: 192
activation: tanh
learning_rate: 0.01
optimizer: rmsprop
Score: 0.21410920470952988

Trial 09 summary
Hyperparameters:
n_neurons_h1: 32
activation: tanh
learning_rate: 0.01
optimizer: adam
Score: 0.2552020400762558

Trial 08 summary
Hyperparameters:
n_neurons_h1: 64
activation: relu
learning_rate: 0.01
optimizer: rmsprop
Score: 0.2750598192214966

Trial 03 summary
Hyperparameters:
n_neurons_h1: 96
activation: tanh
learning_rate: 0.0001
optimizer: rmsprop
Score: 0.4811141937971115

Trial 06 summary
Hyperparameters:
n_neurons_h1: 224
activation: relu
learning_rate: 0.01
optimizer: sgd
Score: 0.6116301119327545

Trial 05 summary
Hyperparameters:
n_neurons_h1: 256
activation: relu
learning_rate: 0.01
optimizer: sgd
Score: 0.6119333803653717

Trial 02 summary
Hyperparameters:
n_neur

# Keras Tuner Final Step: Build a New Model with the Best Hyperparameters
Now that we have found the best parameters, how do we get a model with **those** parameters?

You ask the tuner for the best set of hyperparameters (best_hps) and then use a special tuner method to build a fresh, compiled model based on those settings.

How to do it:

1) Get the **best_hps** object as done in the code above

2) Call tuner.hypermodel.build(best_hps). This takes the best parameters and passes them to your build_model_tuner function to construct the final model.   **NOTE: This is just the model structure - you still have to train it!**   But now you can train it for me epochs (if necessary) and then save this best model to disk for use  later.

We show how to do this below.

In [58]:

# Get the optimal hyperparameters (just redo step from above)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameter configuration
final_model = tuner.hypermodel.build(best_hps)

#
# If we reload this right before fitting the model, the model will start from scratch
final_model.save_weights('final_model_init.weights.h5')


# It's good practice to check the architecture
print("\n--- Final Model Summary ---")
final_model.summary()

print("\n REMEMBER: This model is nto trained yet!!")



--- Final Model Summary ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 192)            │       150,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,930 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 152,650 (596.29 KB)

 Trainable params: 152,650 (596.29 KB)

 Non-trainable params: 0 (0.00 B)


 REMEMBER: This model is nto trained yet!!


# Exercise 5: Now train the best model

Train the model for a maximum of 50 epochs:
- Use EarlyStopping
- Use ModelCheckpoint.  IMPORTANT: use filepath='final_tuned_model_inclass.keras' since you will need this for the assignment!
- Make sure you call "final_model.load_weights('final_model_init.weights.h5')" **before** fitting....
- Also, use the variable "final_results" to store your result history, so we don't confuse it with the above results.

Make sure you also:
- plot the accuracy and loss for test and train, versus epoch
- print out the confusion matrix

In [59]:

#
# Define the EarlyStopping and ModelCheckpoint callbacks
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=10),
             keras.callbacks.ModelCheckpoint(filepath='final_tuned_model_inclass.keras', 
                                            monitor='val_loss', 
                                            save_best_only=True)]
#
# Now load the initial weights we saved above - this **ensures** the model starts training from scratch
final_model.load_weights('final_model_init.weights.h5')

#
# Actually fit the model
final_results = final_model.fit(train_images_temp, 
                                train_labels_cat_temp,
                                epochs=50,
                                batch_size=128,
                                verbose=1,
                                callbacks=callbacks,
                                validation_data=(val_images, val_labels_cat))

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6363 - loss: 1.4255 - val_accuracy: 0.8029 - val_loss: 0.6095
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8835 - loss: 0.3607 - val_accuracy: 0.8643 - val_loss: 0.4295
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9213 - loss: 0.2457 - val_accuracy: 0.9300 - val_loss: 0.2630
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9490 - loss: 0.1549 - val_accuracy: 0.8100 - val_loss: 0.7319
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9547 - loss: 0.1403 - val_accuracy: 0.9343 - val_loss: 0.2607
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9736 - loss: 0.0758 - val_accuracy: 0.9257 - val_loss: 0.2977
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9855 - loss: 0.0508 - val_accuracy: 0.9371 - val_loss: 0.2773
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9877 - loss: 0.0428 - val_accuracy: 0.9329 - val_loss:

In [60]:
print("total notebook time:",time.time()-t00)

total notebook time: 76.13888430595398


In [61]:
df = pd.DataFrame(final_results.history)
df['epoch'] = df.index + 1
display(df.style)

fig = px.line(df, x='epoch', y=['accuracy','val_accuracy'], title='Accuracy vs Epoch')
fig.show("plotly_mimetype")

fig = px.line(df, x='epoch', y=['loss','val_loss'], title='Loss vs Epoch')
fig.show("plotly_mimetype")

,accuracy,loss,val_accuracy,val_loss,epoch
0,0.771746,0.857437,0.802857,0.609467,1
1,0.892857,0.348092,0.864286,0.429489,2
2,0.926984,0.226924,0.930000,0.263043,3
3,0.948095,0.160058,0.810000,0.731941,4
4,0.956667,0.129915,0.934286,0.260707,5
5,0.970159,0.088358,0.925714,0.297734,6
6,0.984762,0.055443,0.937143,0.277335,7
7,0.989048,0.039038,0.932857,0.342945,8
8,0.989206,0.035849,0.940000,0.302020,9
9,0.993810,0.018974,0.935714,0.296423,10
